## **Principal Component Analysis (PCA) - Part 2: Projection & Scikit-Learn**

### Topic Roadmap

**1. Imports & Setup (Recap from Part 1)**

**2. Selecting Principal Components**

**3. Transforming (Projecting) the Data**

**4. Visualizing the Result (2D)**

**5. Scikit-Learn Implementation**

**6. Key Revision Notes**

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

In [7]:
# 1. Generate Data
np.random.seed(23)
class1 = np.random.multivariate_normal([0, 0, 0], [[1, 0, 0], [0, 1, 0], [0, 0, 1]], 20)
class2 = np.random.multivariate_normal([1, 1, 1], [[1, 0, 0], [0, 1, 0], [0, 0, 1]], 20)

df_class1 = pd.DataFrame(class1, columns=['feature1', 'feature2', 'feature3'])
df_class1['target'] = 1
df_class2 = pd.DataFrame(class2, columns=['feature1', 'feature2', 'feature3'])
df_class2['target'] = 0

df = pd.concat([df_class1, df_class2], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# 2. Scale Data
X_scaled = StandardScaler().fit_transform(df[['feature1', 'feature2', 'feature3']])

# 3. Compute Covariance & Eigenvectors
covariance_matrix = np.cov(X_scaled.T)
eigen_values, eigen_vectors = np.linalg.eig(covariance_matrix)

### **2. Selecting Principal Components**

To reduce dimensionality from 3D to 2D, we must select the top 2 eigenvectors. These correspond to the largest eigenvalues, as they capture the most variance

In [8]:
# Sort eigenvalues in descending order and get their indices
sorted_indices = np.argsort(eigen_values)[::-1]
sorted_eigenvalues = eigen_values[sorted_indices]

# Reorder eigenvectors based on sorted eigenvalues (Note: Eigenvectors are columns)
sorted_eigenvectors = eigen_vectors[:, sorted_indices]

In [9]:
# Select the top 2 eigenvectors (Principal Components)
top_n_components = 2
principal_components = sorted_eigenvectors[:, :top_n_components]
print("Top 2 Principal Components (Weight Matrix):\n", principal_components)

Top 2 Principal Components (Weight Matrix):
 [[-0.53875915 -0.69363291]
 [-0.65608325 -0.01057596]
 [-0.52848211  0.72025103]]


### **3. Transforming (Projecting) the Data**

We transform our original scaled dataset into the new 2D subspace using the dot product: $$X_{new} = X_{scaled} \cdot W$$
Where $W$ is the weight matrix composed of our selected eigenvectors

In [10]:
# Project data via dot product
transformed_data = np.dot(X_scaled, principal_components)

In [11]:
# Convert to DataFrame for visualization
pca_df = pd.DataFrame(transformed_data, columns=['PC1', 'PC2'])
pca_df['target'] = df['target'].astype(str)
pca_df.head()

,PC1,PC2,target
0,0.817930,1.203797,1
1,1.403371,0.029261,1
2,0.216358,-0.197431,1
3,-0.274866,-0.184906,0
4,1.098223,-1.011722,1


### **4. Visualizing the Result (2D)**

By projecting the 3D data onto a 2D plane, we can now visualize the class separation using Plotly

In [12]:
import plotly.express as px

fig = px.scatter(
    x=pca_df['PC1'],
    y=pca_df['PC2'],
    color=pca_df['target'],
    color_discrete_sequence=px.colors.qualitative.G10,
    labels={'x': 'Principal Component 1', 'y': 'Principal Component 2'},
    title='2D PCA Projection'
)

fig.update_traces(marker=dict(size=12, line=dict(width=2, color='DarkSlateGrey')))
fig.show()

### **5. Scikit-Learn Implementation**

In practice, the entire mathematical process (Centering, Covariance, Eigen Decomposition, and Projection) is handled efficiently via `sklearn.decomposition.PCA`.

In [13]:
from sklearn.decomposition import PCA

# Initialize PCA and specify the number of components
pca = PCA(n_components=2)

In [14]:
# Fit the model and transform the scaled data simultaneously
sklearn_pca_data = pca.fit_transform(X_scaled)

In [15]:
# Create DataFrame for the Scikit-Learn results
sklearn_pca_df = pd.DataFrame(sklearn_pca_data, columns=['PC1', 'PC2'])
sklearn_pca_df['target'] = df['target'].values
sklearn_pca_df.head()

,PC1,PC2,target
0,-0.817930,1.203797,1
1,-1.403371,0.029261,1
2,-0.216358,-0.197431,1
3,0.274866,-0.184906,0
4,-1.098223,-1.011722,1


In [16]:
print("Explained Variance Ratio by PC1 and PC2:", pca.explained_variance_ratio_)

Explained Variance Ratio by PC1 and PC2: [0.43992211 0.30731052]


### **6. Key Revision Notes**

- **Sorting Eigenvalues:** Essential because `np.linalg.eig` does not guarantee ordered output. You must pair the highest eigenvalues with their respective eigenvectors to capture maximum variance.
- **Matrix Multiplication (Projection):** Done via the dot product of the original scaled dataset matrix and the top-$k$ eigenvector matrix.
- **Scikit-Learn Workflow:** `PCA(n_components=k)` replaces all manual linear algebra steps. Always pass **Scaled Data** to `pca.fit_transform()`.
- **Explained Variance Ratio:** Provided by Scikit-Learn's `pca.explained_variance_ratio_`, indicating the percentage of the dataset's total variance captured by each principal component.